# Population by Elevation Companion Analysis

This companion notebook supports the repository-level hypsographic demography summary and animation. It is not a core manuscript figure notebook. It uses the integer-elevation age-sex population fact table to summarize global population below selected elevation thresholds and to animate 2025 population pyramids as the elevation threshold descends globally and by continent.

In [ ]:
from pathlib import Path

import pandas as pd

from IPython.display import display, Image, Markdown


ROOT = Path.cwd()
if not (ROOT / "data" / "dataset_s1_hypsographic_demography.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "data" / "dataset_s1_hypsographic_demography.csv").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")


DATA = ROOT / "data"
FIG = ROOT / "assets"
TAB = ROOT / "outputs" / "population_by_elevation"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)


INPUT_GLOBAL_ELEVATION_AGE_SEX = DATA / "fact_population_by_integer_elevation_age_sex_2015_2025.parquet"
YEARS = [2015, 2025]
OUT_TABLE = TAB / "population_by_elevation_threshold_summary.csv"
OUT_AGE_TABLE = TAB / "population_by_elevation_threshold_age_summary.csv"
OUT_GIF = FIG / "population_by_elevation_pyramid.gif"
OUT_CONTINENT_PANEL_GIF = FIG / "population_by_elevation_continents" / "population_by_elevation_pyramid_continents_6panel.gif"


YEAR_COL = "year"
GEOGRAPHY_COL = "geography_level"
CONTINENT_COL = "continent"
ELEV_COL = "elevation_m"
BROAD_AGE_COL = "broad_age_group"
POP_COL = "population_count"


BROAD_AGE_LABELS = {
    "young_0_14": "Young (0-14)",
    "working_age_15_64": "Working age (15-64)",
    "old_age_65_plus": "Older (65+)",
    "older_65_plus": "Older (65+)",
}
BROAD_AGE_ORDER = ["young_0_14", "working_age_15_64", "old_age_65_plus", "older_65_plus"]


THRESHOLDS_M = [100, 150, 200, 500, 1000, 1500, 3500]


def load_global_year(year: int) -> pd.DataFrame:
    path = INPUT_GLOBAL_ELEVATION_AGE_SEX
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path.relative_to(ROOT)}")
    df = pd.read_parquet(path)
    if YEAR_COL in df.columns:
        df = df[df[YEAR_COL].astype(int).eq(year)].copy()
    if GEOGRAPHY_COL in df.columns:
        df = df[df[GEOGRAPHY_COL].astype(str).eq("global")].copy()
    if CONTINENT_COL in df.columns:
        df = df[df[CONTINENT_COL].astype(str).eq("Global")].copy()
    df[ELEV_COL] = pd.to_numeric(df[ELEV_COL], errors="coerce")
    df[POP_COL] = pd.to_numeric(df[POP_COL], errors="coerce").fillna(0)
    df = df.dropna(subset=[ELEV_COL])
    if df.empty:
        raise ValueError(f"No global integer-elevation rows found for {year}.")
    return df


print("Repo root:", ROOT)


## Threshold Summary

The threshold table includes 150 m because the headline companion result is that roughly half of the global population lives at or below 150 m elevation.

In [ ]:
threshold_rows = []
threshold_age_rows = []
loaded = {}

for year in YEARS:
    df = load_global_year(year)
    loaded[year] = df
    total_pop = df[POP_COL].sum()
    print(f"{year}: {len(df):,} global rows; total population {total_pop:,.0f}")

    for threshold in THRESHOLDS_M:
        pop_below = df.loc[df[ELEV_COL] <= threshold, POP_COL].sum()
        pop_above = total_pop - pop_below
        threshold_rows.append({
            "year": year,
            "threshold_m": threshold,
            "population_below_or_equal": pop_below,
            "percent_below_or_equal": pop_below / total_pop * 100,
            "population_above": pop_above,
            "percent_above": pop_above / total_pop * 100,
        })

    age_totals = df.groupby(BROAD_AGE_COL, as_index=False)[POP_COL].sum()
    for age_row in age_totals.itertuples(index=False):
        broad_age_group = getattr(age_row, BROAD_AGE_COL)
        age_total = getattr(age_row, POP_COL)
        age_df = df[df[BROAD_AGE_COL].eq(broad_age_group)]

        for threshold in THRESHOLDS_M:
            pop_below = age_df.loc[age_df[ELEV_COL] <= threshold, POP_COL].sum()
            pop_above = age_total - pop_below
            threshold_age_rows.append({
                "year": year,
                "threshold_m": threshold,
                "broad_age_group": broad_age_group,
                "age_group_label": BROAD_AGE_LABELS.get(broad_age_group, broad_age_group),
                "population_below_or_equal": pop_below,
                "percent_below_or_equal_within_age_group": pop_below / age_total * 100,
                "population_above": pop_above,
                "percent_above_within_age_group": pop_above / age_total * 100,
                "age_group_total_population": age_total,
            })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv(OUT_TABLE, index=False)

threshold_age_summary = pd.DataFrame(threshold_age_rows)
threshold_age_summary["broad_age_sort"] = threshold_age_summary["broad_age_group"].map(
    {age: i for i, age in enumerate(BROAD_AGE_ORDER)}
).fillna(len(BROAD_AGE_ORDER))
threshold_age_summary = (
    threshold_age_summary
    .sort_values(["year", "threshold_m", "broad_age_sort", "broad_age_group"])
    .drop(columns="broad_age_sort")
)
threshold_age_summary.to_csv(OUT_AGE_TABLE, index=False)

below_150 = threshold_summary[threshold_summary["threshold_m"].eq(150)].copy()
below_150_by_age = threshold_age_summary[threshold_age_summary["threshold_m"].eq(150)].copy()
print(f"Saved {OUT_TABLE.relative_to(ROOT)}")
print(f"Saved {OUT_AGE_TABLE.relative_to(ROOT)}")

display(
    threshold_summary.style.format({
        "threshold_m": "{:,.0f}",
        "population_below_or_equal": "{:,.0f}",
        "percent_below_or_equal": "{:.2f}%",
        "population_above": "{:,.0f}",
        "percent_above": "{:.2f}%",
    })
)

display(
    below_150_by_age.style.format({
        "threshold_m": "{:,.0f}",
        "population_below_or_equal": "{:,.0f}",
        "percent_below_or_equal_within_age_group": "{:.2f}%",
        "population_above": "{:,.0f}",
        "percent_above_within_age_group": "{:.2f}%",
        "age_group_total_population": "{:,.0f}",
    })
)

display(Markdown(
    "**Key result:** in 2025, "
    f"{below_150.loc[below_150['year'].eq(2025), 'percent_below_or_equal'].iloc[0]:.1f}% "
    "of the global population lived at or below 150 m elevation "
    f"({below_150.loc[below_150['year'].eq(2015), 'percent_below_or_equal'].iloc[0]:.1f}% in 2015). "
    "By broad age group in 2025, the corresponding shares were "
    + ", ".join(
        f"{row.age_group_label}: {row.percent_below_or_equal_within_age_group:.1f}%"
        for row in below_150_by_age[below_150_by_age["year"].eq(2025)].itertuples(index=False)
    )
    + "."
))


## Population-By-Elevation GIF

The production animation is generated by `processing/07_plot_population_elevation_pyramid_gif.py`, which is the single GIF renderer used by the public workflow and README. It reads the optional local input `data/fact_population_by_integer_elevation_age_sex_2015_2025.parquet`, writes all GIFs under `assets/`, and writes diagnostic threshold tables to the ignored directory `outputs/population_by_elevation/`.


In [ ]:
import runpy


runpy.run_path(str(ROOT / "processing" / "07_plot_population_elevation_pyramid_gif.py"), run_name="__main__")
display(Markdown(
    "Animations saved to:\n"
    f"- `{OUT_GIF.relative_to(ROOT)}`\n"
    f"- `{OUT_CONTINENT_PANEL_GIF.relative_to(ROOT)}`"
))
display(Image(filename=str(OUT_CONTINENT_PANEL_GIF)))
